In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings 
warnings.filterwarnings("ignore")

df = pd.read_csv('../data/processed/telco_cleaned.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)


df = df.dropna()

print("NaNs left:", df.isnull().sum().sum())
print("Shape:", df.shape)


df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

X = df.drop('Churn', axis=1)
y = df['Churn']

NaNs left: 0
Shape: (7043, 20)


In [16]:
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

Numeric: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [17]:
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
print("Shape after encoding:", X.shape)

Shape after encoding: (7043, 30)


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (5634, 30) Test: (1409, 30)


In [19]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

IMPORT MODELS AND METRICS

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix)

In [21]:
def evaluate(name, model, X_test, y_test):
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print(f"Accuracy : {accuracy_score(y_test, preds):.4f}")
    print(f"Precision: {precision_score(y_test, preds):.4f}")
    print(f"Recall   : {recall_score(y_test, preds):.4f}")
    print(f"F1-Score : {f1_score(y_test, preds):.4f}")
    print(f"ROC-AUC  : {roc_auc_score(y_test, probs):.4f}")

In [25]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
evaluate("Logistic Regression", lr, X_test_scaled, y_test)


Logistic Regression
Accuracy : 0.8070
Precision: 0.6584
Recall   : 0.5668
F1-Score : 0.6092
ROC-AUC  : 0.8418


In [26]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
evaluate("Random Forest", rf, X_test_scaled, y_test)


Random Forest
Accuracy : 0.7850
Precision: 0.6187
Recall   : 0.4947
F1-Score : 0.5498
ROC-AUC  : 0.8248


In [27]:
xgb = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb.fit(X_train_scaled, y_train)
evaluate("XGBoost", xgb, X_test_scaled, y_test)


XGBoost
Accuracy : 0.7764
Precision: 0.5891
Recall   : 0.5214
F1-Score : 0.5532
ROC-AUC  : 0.8183


In [28]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1]
}

grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_grid, cv=5, scoring='roc_auc', n_jobs=-1
)
grid.fit(X_train_scaled, y_train)

print("Best params:", grid.best_params_)
print("Best CV ROC-AUC:", grid.best_score_)

best_model = grid.best_estimator_
evaluate("Tuned XGBoost", best_model, X_test_scaled, y_test)

Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
Best CV ROC-AUC: 0.8466879997544616

Tuned XGBoost
Accuracy : 0.7991
Precision: 0.6553
Recall   : 0.5134
F1-Score : 0.5757
ROC-AUC  : 0.8451


In [29]:
import joblib

joblib.dump(best_model, '../models/churn_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(X.columns.tolist(), '../models/feature_columns.pkl')

print("Model saved ✅")

Model saved ✅


In [10]:
df.isnull().sum()[df.isnull().sum() > 0]

TotalCharges    11
dtype: int64

In [11]:
X.isnull().sum()[X.isnull().sum() > 0]

TotalCharges    11
dtype: int64

In [12]:
import numpy as np
print("NaNs in X_train_scaled:", np.isnan(X_train_scaled).sum())
print("NaNs in X_test_scaled:", np.isnan(X_test_scaled).sum())

NaNs in X_train_scaled: 8
NaNs in X_test_scaled: 3


In [13]:
X.columns[X.isnull().any()].tolist()

['TotalCharges']

In [14]:
df['TotalCharges'].isnull().sum()

np.int64(11)

In [22]:
from sklearn.impute import SimpleImputer


scaler = StandardScaler()
imputer = SimpleImputer(strategy='median') 

X_train_scaled = imputer.fit_transform(X_train)
X_train_scaled = scaler.fit_transform(X_train_scaled)

X_test_scaled = imputer.transform(X_test)
X_test_scaled = scaler.transform(X_test_scaled)

In [23]:

import numpy as np
print("NaNs in X_train_scaled:", np.isnan(X_train_scaled).sum())
print("NaNs in X_test_scaled:", np.isnan(X_test_scaled).sum())

NaNs in X_train_scaled: 0
NaNs in X_test_scaled: 0
